# COMPARACION ConvVAE vs ConvHQVAE sobre CIFAR-100 — Ruido mixto fijo: Gaussian + Salt-Pepper

Version para **CIFAR-100** del notebook homonimo de MNIST (`test-hqvae/vcae_VS_hqvae_gaussian_saltpepper.ipynb`).
Se mantiene exactamente la misma pipeline: mismos modelos `ConvVAE` / `ConvHQVAE`,
mismo circuito cuantico de 6 qubits, mismo protocolo de entrenamiento
(`latent_dim=12`, 15 epocas, `beta=0.5`, Adam para el clasico y AdamW para el
hibrido) y las mismas metricas. La unica variable que cambia respecto al
notebook de MNIST es **el contenido de las imagenes**.

Cada imagen recibe siempre el mismo tipo de ruido mixto: **Gaussian + Salt-Pepper**.

Split (identico al de MNIST): 300 train / 100 val / 100 test.

> **Nota sobre por que esto importa.** En MNIST cerca del 80 % de los pixeles
> son fondo negro exacto ($x = 0$), donde el speckle ($x + x\eta$) no tiene
> ningun efecto y la sal-y-pimienta resulta muy visible. Ese contraste ofrece
> un atajo que no existe en imagenes naturales. CIFAR-100 elimina ese atajo y
> permite comprobar si los resultados obtenidos sobre MNIST se sostienen.

## 1. Configuracion, datos y modelo de ruido

### 1.1. Imports

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset, random_split

### 1.2. Configuracion del dataset

In [ ]:
# ---------------------------------------------------------------------
# CONFIGURACION DEL DATASET
# ---------------------------------------------------------------------
# Estas constantes son el unico punto que hay que tocar para pasar de
# la variante "CIFAR nativo" a la variante "CIFAR emparejado con MNIST".
#
#   IMG_SIZE = 32  -> resolucion nativa de CIFAR-100 (variante por defecto)
#   IMG_SIZE = 28  -> se redimensiona a 28x28, de modo que TODAS las
#                     dimensiones internas coinciden exactamente con las de
#                     los notebooks de MNIST y la unica variable que cambia
#                     entre datasets es el contenido de la imagen.
#
#   GRAYSCALE = True  -> 1 canal. Recomendado: el modelo de ruido de este TFM
#                        es por pixel y esta definido sobre imagenes en escala
#                        de grises; ademas mantiene identica la arquitectura.
#   GRAYSCALE = False -> 3 canales RGB. Requiere la mascara compartida de
#                        sal-y-pimienta (ya implementada mas abajo).
#
#   NOISE_SCALE -> factor que multiplica los tres parametros de ruido.
#                  Los valores base (0.25 / 0.15 / 0.35) se ajustaron para
#                  MNIST, donde ~82 % de los pixeles son fondo negro exacto
#                  y el speckle (x + x*eta) no tiene ningun efecto sobre el.
#                  En CIFAR-100 solo ~1 % de los pixeles esta cerca de cero,
#                  asi que esos mismos valores dan SNR negativa (el ruido
#                  tiene mas potencia que la senal) y la tarea es irresoluble.
#                  Escalados medidos sobre 300 imagenes:
#                      1.0  -> SNR media -0.83 dB   (MNIST: +1.61 dB)
#                      0.6  -> SNR media +2.20 dB   equipara la dificultad
#                      0.4  -> SNR media +4.71 dB   deja mas estructura visible
# ---------------------------------------------------------------------
IMG_SIZE = 32
GRAYSCALE = True
NOISE_SCALE = 0.4

IN_CH = 1 if GRAYSCALE else 3

# Tras dos convoluciones stride=2 el mapa espacial queda en IMG_SIZE // 4.
FEAT = IMG_SIZE // 4

# Dimension aplanada del encoder del VAE (64 canales) y del clasificador (32).
ENC_FLAT = 64 * FEAT * FEAT
CLF_FLAT = 32 * FEAT * FEAT

# Parametros de ruido efectivos.
SIGMA_GAUSS   = 0.25 * NOISE_SCALE
PROB_SP       = 0.15 * NOISE_SCALE
SIGMA_SPECKLE = 0.35 * NOISE_SCALE

print(f"Imagen      : {IN_CH}x{IMG_SIZE}x{IMG_SIZE}")
print(f"Mapa conv   : {FEAT}x{FEAT}")
print(f"Flatten VAE : {ENC_FLAT}   (MNIST 28x28 -> 3136)")
print(f"Flatten CLF : {CLF_FLAT}   (MNIST 28x28 -> 1568)")
print(f"Ruido       : escala {NOISE_SCALE}  -> sigma_g={SIGMA_GAUSS:.3f} "
      f"p={PROB_SP:.3f} sigma_s={SIGMA_SPECKLE:.3f}")

### 1.3. Reproducibilidad (semilla)

In [ ]:
import random
import os
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seed fijada: {SEED}")

### 1.4. Carga de CIFAR-100 y submuestreo

In [ ]:
# CIFAR-100 se convierte a escala de grises (si GRAYSCALE) y se redimensiona a
# IMG_SIZE antes de pasar a tensor en [0, 1], que es el rango que asume el
# modelo de ruido.
_tf = []
if GRAYSCALE:
    _tf.append(transforms.Grayscale(num_output_channels=1))
if IMG_SIZE != 32:
    _tf.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))
_tf.append(transforms.ToTensor())

transform = transforms.Compose(_tf)

X_train_full = datasets.CIFAR100(root="./data", train=True,  download=True, transform=transform)
X_test_full  = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

In [ ]:
n_samples_train = 400
n_samples_test = 100

# En los notebooks de MNIST el submuestreo se hacia recortando `.data` y
# `.targets`. En CIFAR-100 `.data` es un ndarray HWC y `.targets` una lista,
# asi que se usa `Subset`, que es equivalente y no depende de la version de
# torchvision. Las etiquetas originales de CIFAR (las 100 categorias) no se
# utilizan: la etiqueta de este problema es el tipo de ruido mixto.
X_train = Subset(X_train_full, range(n_samples_train))
X_test  = Subset(X_test_full,  range(n_samples_test))

In [ ]:
train_size = 300
val_size = 100

split_generator = torch.Generator()
split_generator.manual_seed(SEED)

train_subset, val_subset = random_split(
    X_train,
    [train_size, val_size],
    generator=split_generator
)

### 1.5. Modelizacion del ruido

Los tres procesos elementales son los mismos que en MNIST. La unica diferencia
esta en `salt_pepper`, cuya mascara se comparte entre canales para que el ruido
impulsivo siga siendo blanco/negro tambien en la variante RGB.

En este notebook la combinacion es **fija**: Gaussian + Salt-Pepper (etiqueta 1).

In [ ]:
def gaussian_noise(img, sigma=None, generator=None):
    sigma = SIGMA_GAUSS if sigma is None else sigma
    noise = torch.randn(img.shape, generator=generator, dtype=img.dtype) * sigma
    return torch.clamp(img + noise, 0, 1)


def salt_pepper(img, prob=None, generator=None):
    """Ruido impulsivo.

    A diferencia de la version de MNIST, la mascara se genera con forma
    (1, H, W) y se difunde sobre los canales. En una imagen RGB una mascara
    independiente por canal produciria pixeles de color aleatorio en lugar de
    impulsos blancos y negros, que es lo que el proceso fisico describe.
    Con 1 canal el comportamiento es identico al de los notebooks de MNIST.
    """
    prob = PROB_SP if prob is None else prob

    noisy = img.clone()

    mask = torch.rand(
        (1, *img.shape[1:]),
        generator=generator,
        dtype=img.dtype
    ).expand_as(img)

    noisy[mask < prob / 2] = 0
    noisy[mask > 1 - prob / 2] = 1

    return noisy


def speckle(img, sigma=None, generator=None):
    sigma = SIGMA_SPECKLE if sigma is None else sigma
    noise = torch.randn(img.shape, generator=generator, dtype=img.dtype) * sigma
    return torch.clamp(img + img * noise, 0, 1)

In [ ]:
def apply_mixed_noise(img, generator):
    # Ruido mixto FIJO para este notebook: Gaussian + Salt-Pepper
    img = gaussian_noise(img, generator=generator)
    img = salt_pepper(img, generator=generator)
    label = 1

    return img, label

In [ ]:
def to_disp(t):
    """Convierte un tensor/array CHW en algo que `imshow` acepte.

    Devuelve HxW si la imagen tiene 1 canal y HxWx3 si tiene 3, de modo que
    las mismas celdas de visualizacion sirven para ambas variantes.
    """
    a = t.detach().cpu().numpy() if torch.is_tensor(t) else np.asarray(t)
    a = np.squeeze(a)
    if a.ndim == 3 and a.shape[0] in (1, 3):
        a = np.transpose(a, (1, 2, 0))
    return np.clip(a, 0, 1)


CMAP = "gray" if IN_CH == 1 else None

### 1.6. Datasets y DataLoaders

In [ ]:
class NoisyCIFARDataset(Dataset):
    def __init__(self, base_dataset, seed):
        self.base = base_dataset
        self.generator = torch.Generator()
        self.generator.manual_seed(seed)

        self.noisy_images = []
        self.labels = []

        for img, _ in self.base:
            noisy_img, label = apply_mixed_noise(img, generator=self.generator)
            self.noisy_images.append(noisy_img)
            self.labels.append(label)

        if len(self.noisy_images) != len(self.labels):
            raise Exception("Incompatible arrays")

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        return self.noisy_images[idx], self.labels[idx]


train_dataset = NoisyCIFARDataset(train_subset, seed=SEED)
val_dataset   = NoisyCIFARDataset(val_subset, seed=SEED + 1)
test_dataset  = NoisyCIFARDataset(X_test, seed=SEED + 2)

In [ ]:
train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=train_generator)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### 1.7. Inspeccion visual del ruido

In [ ]:
imgs, _ = next(iter(train_loader))
clean_imgs = torch.stack([train_subset[i][0] for i in range(8)])

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for i in range(8):
    axes[0, i].imshow(to_disp(clean_imgs[i]), cmap=CMAP)
    axes[0, i].axis("off")
    axes[1, i].imshow(to_disp(imgs[i]), cmap=CMAP)
    axes[1, i].axis("off")

axes[0, 0].set_title("Limpia", loc="left", fontsize=10)
axes[1, 0].set_title("Con ruido", loc="left", fontsize=10)
plt.suptitle("CIFAR-100: imagen limpia (arriba) y con ruido (abajo)")
plt.tight_layout()
plt.show()

## 2. Denoising: ConvVAE clasico vs ConvHQVAE hibrido

El objetivo pasa a ser reconstruir la imagen limpia a partir de la ruidosa.
El encoder es convolucional y, en el modelo hibrido, el espacio latente se
procesa mediante un QNN antes de decodificar.

### 2.1. Dataset de denoising

In [ ]:
class DenoisingCIFARDataset(Dataset):
    def __init__(self, base_dataset, seed):
        self.base = base_dataset
        self.generator = torch.Generator()
        self.generator.manual_seed(seed)

        self.noisy_images = []
        self.clean_images = []

        for img, _ in self.base:
            noisy_img, _ = apply_mixed_noise(img, generator=self.generator)
            self.clean_images.append(img)
            self.noisy_images.append(noisy_img)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        return self.noisy_images[idx], self.clean_images[idx]


train_dataset_dn = DenoisingCIFARDataset(train_subset, seed=SEED)
val_dataset_dn   = DenoisingCIFARDataset(val_subset, seed=SEED + 1)
test_dataset_dn  = DenoisingCIFARDataset(X_test, seed=SEED + 2)

train_generator_dn = torch.Generator()
train_generator_dn.manual_seed(SEED)

train_loader_dn = DataLoader(train_dataset_dn, batch_size=16, shuffle=True, generator=train_generator_dn)
val_loader_dn   = DataLoader(val_dataset_dn, batch_size=16, shuffle=False)
test_loader_dn  = DataLoader(test_dataset_dn, batch_size=16, shuffle=False)

### 2.2. Bloque cuantico del espacio latente (6 qubits)

In [ ]:
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit.circuit.library import real_amplitudes, zz_feature_map
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_machine_learning.connectors import TorchConnector
from qiskit.quantum_info import SparsePauliOp

estimator = Estimator()

observables = [  # correlacion en anillo
    SparsePauliOp.from_list([("ZZIIII", 1.0)]),  # q0-q1
    SparsePauliOp.from_list([("IZZIII", 1.0)]),  # q1-q2
    SparsePauliOp.from_list([("IIZZII", 1.0)]),  # q2-q3
    SparsePauliOp.from_list([("IIIZZI", 1.0)]),  # q3-q4
    SparsePauliOp.from_list([("IIIIZZ", 1.0)]),  # q4-q5
    SparsePauliOp.from_list([("ZIIIIZ", 1.0)]),  # q5-q0
]


def create_qnn_vae():
    n_qubits = 6
    feature_map = zz_feature_map(n_qubits, reps=1, entanglement="full")
    ansatz = real_amplitudes(n_qubits, entanglement="linear", reps=1)

    qc = QuantumCircuit(n_qubits)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        input_gradients=True,
        estimator=estimator,
        observables=observables,
    )
    return qnn


qnn_vae = create_qnn_vae()

### 2.3. Arquitecturas

Ambos modelos comparten encoder y decoder. El `ConvVAE` sustituye el bloque
cuantico por un `latent_transform` clasico de la misma dimension de salida,
de modo que las dos arquitecturas son comparables una a una.

- Encoder: `Conv2d(IN_CH -> 32 -> 64)` + `Flatten`.
- Latente: `fc_mu`, `fc_logvar` (`latent_dim = 12`).
- Bloque intermedio: QNN de 6 qubits (hibrido) o MLP (clasico).
- Decoder: `ConvTranspose2d` hasta reconstruir `IN_CH x IMG_SIZE x IMG_SIZE`.

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=12):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(IN_CH, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )

        self.fc_mu = nn.Linear(ENC_FLAT, latent_dim)
        self.fc_logvar = nn.Linear(ENC_FLAT, latent_dim)

        # Bloque clasico equivalente al bloque cuantico del modelo hibrido:
        # misma entrada (latente) y misma dimension de salida (len(observables)).
        self.latent_transform = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, len(observables))
        )

        self.fc_decode = nn.Linear(latent_dim + len(observables), ENC_FLAT)

        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, FEAT, FEAT)),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, IN_CH, 4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparametrize(self, mu, logvar):
        sigma = torch.exp(0.5 * logvar)
        return mu + sigma * torch.randn_like(sigma)

    def decode(self, z):
        z_classical = self.latent_transform(z)
        z_combined = torch.cat([z, z_classical], dim=1)
        return self.decoder(self.fc_decode(z_combined))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        return self.decode(z), mu, logvar

    def reconstruct(self, x):
        mu, logvar = self.encode(x)
        return self.decode(mu)

In [ ]:
class ConvHQVAE(nn.Module):
    def __init__(self, latent_dim=12, qnn=None):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(IN_CH, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )

        self.fc_mu = nn.Linear(ENC_FLAT, latent_dim)
        self.fc_logvar = nn.Linear(ENC_FLAT, latent_dim)

        self.qnn = TorchConnector(qnn) if qnn is not None else nn.Identity()

        # Proyeccion clasica latent_dim -> n de qubits de entrada del QNN.
        # Con latent_dim=12 y 6 qubits es necesaria; si coincidieran seria
        # una Identity y no cambiaria nada.
        n_qnn_inputs = qnn.num_inputs if qnn is not None else latent_dim
        self.to_qnn = (
            nn.Linear(latent_dim, n_qnn_inputs)
            if latent_dim != n_qnn_inputs else nn.Identity()
        )

        self.fc_decode = nn.Linear(latent_dim + len(observables), ENC_FLAT)

        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, FEAT, FEAT)),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, IN_CH, 4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparametrize(self, mu, logvar):
        sigma = torch.exp(0.5 * logvar)
        return mu + sigma * torch.randn_like(sigma)

    def decode(self, z):
        z_qnn = self.qnn(self.to_qnn(z))
        z_combined = torch.cat([z, z_qnn], dim=1)
        return self.decoder(self.fc_decode(z_combined))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        return self.decode(z), mu, logvar

    def reconstruct(self, x):
        mu, logvar = self.encode(x)
        return self.decode(mu)

### 2.4. Funcion de perdida ($\beta$-VAE)

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=0.5):
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    total = bce + beta * kl
    return total, bce, kl

### 2.5. Entrenamiento del ConvVAE clasico

In [ ]:
model_cvae = ConvVAE(latent_dim=12).to(device)
optimizer_cvae = optim.Adam(model_cvae.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 15

cvae_train_loss, cvae_val_loss = [], []
cvae_train_bce, cvae_val_bce = [], []
cvae_train_kl, cvae_val_kl = [], []

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    model_cvae.train()
    train_loss = train_bce = train_kl = 0
    n_train = 0

    for noisy, clean in train_loader_dn:
        noisy, clean = noisy.to(device), clean.to(device)

        optimizer_cvae.zero_grad()
        recon, mu, logvar = model_cvae(noisy)
        loss, bce, kl = vae_loss(recon, clean, mu, logvar)
        loss.backward()
        optimizer_cvae.step()

        bs = noisy.size(0)
        train_loss += loss.item() * bs
        train_bce += bce.item() * bs
        train_kl += kl.item() * bs
        n_train += bs

    train_loss /= n_train
    train_bce /= n_train
    train_kl /= n_train

    model_cvae.eval()
    val_loss = val_bce = val_kl = 0
    n_val = 0

    with torch.no_grad():
        for noisy, clean in val_loader_dn:
            noisy, clean = noisy.to(device), clean.to(device)
            recon, mu, logvar = model_cvae(noisy)
            loss, bce, kl = vae_loss(recon, clean, mu, logvar)

            bs = noisy.size(0)
            val_loss += loss.item() * bs
            val_bce += bce.item() * bs
            val_kl += kl.item() * bs
            n_val += bs

    val_loss /= n_val
    val_bce /= n_val
    val_kl /= n_val

    cvae_train_loss.append(train_loss); cvae_val_loss.append(val_loss)
    cvae_train_bce.append(train_bce);   cvae_val_bce.append(val_bce)
    cvae_train_kl.append(train_kl);     cvae_val_kl.append(val_kl)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Train BCE: {train_bce:.4f} | Val BCE: {val_bce:.4f} | "
        f"Train KL: {train_kl:.4f} | Val KL: {val_kl:.4f}"
    )

training_time = time.time() - start_time
print(f"[ConvVAE] Tiempo de entrenamiento: {training_time:.2f} segundos")
print(f"Tiempo en minutos y segundos: {int(training_time // 60)} minutos y {int(training_time % 60)} segundos")

### 2.6. Entrenamiento del ConvHQVAE hibrido

In [ ]:
model_hqvae = ConvHQVAE(latent_dim=12, qnn=qnn_vae).to(device)
optimizer = optim.AdamW(model_hqvae.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 15

hqvae_train_loss, hqvae_val_loss = [], []
hqvae_train_bce, hqvae_val_bce = [], []
hqvae_train_kl, hqvae_val_kl = [], []

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    model_hqvae.train()
    train_loss = train_bce = train_kl = 0
    n_train = 0

    for noisy, clean in train_loader_dn:
        noisy, clean = noisy.to(device), clean.to(device)

        optimizer.zero_grad()
        recon, mu, logvar = model_hqvae(noisy)
        loss, bce, kl = vae_loss(recon, clean, mu, logvar)
        loss.backward()
        optimizer.step()

        bs = noisy.size(0)
        train_loss += loss.item() * bs
        train_bce += bce.item() * bs
        train_kl += kl.item() * bs
        n_train += bs

    train_loss /= n_train
    train_bce /= n_train
    train_kl /= n_train

    model_hqvae.eval()
    val_loss = val_bce = val_kl = 0
    n_val = 0

    with torch.no_grad():
        for noisy, clean in val_loader_dn:
            noisy, clean = noisy.to(device), clean.to(device)
            recon, mu, logvar = model_hqvae(noisy)
            loss, bce, kl = vae_loss(recon, clean, mu, logvar)

            bs = noisy.size(0)
            val_loss += loss.item() * bs
            val_bce += bce.item() * bs
            val_kl += kl.item() * bs
            n_val += bs

    val_loss /= n_val
    val_bce /= n_val
    val_kl /= n_val

    hqvae_train_loss.append(train_loss); hqvae_val_loss.append(val_loss)
    hqvae_train_bce.append(train_bce);   hqvae_val_bce.append(val_bce)
    hqvae_train_kl.append(train_kl);     hqvae_val_kl.append(val_kl)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Train BCE: {train_bce:.4f} | Val BCE: {val_bce:.4f} | "
        f"Train KL: {train_kl:.4f} | Val KL: {val_kl:.4f}"
    )

training_time = time.time() - start_time
print(f"[ConvHQVAE] Tiempo de entrenamiento: {training_time:.2f} segundos")
print(f"Tiempo en minutos y segundos: {int(training_time // 60)} minutos y {int(training_time % 60)} segundos")

## 3. Resultados

### 3.1. Curvas de perdida

In [ ]:
epochs_range = range(1, len(cvae_val_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs_range, cvae_train_loss, marker="o", label="ConvVAE train")
axes[0].plot(epochs_range, cvae_val_loss, marker="o", label="ConvVAE val")
axes[0].plot(epochs_range, hqvae_train_loss, marker="s", label="ConvHQVAE train")
axes[0].plot(epochs_range, hqvae_val_loss, marker="s", label="ConvHQVAE val")
axes[0].set_title("Perdida total")
axes[0].set_xlabel("Epoca"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, cvae_val_bce, marker="o", label="ConvVAE BCE")
axes[1].plot(epochs_range, hqvae_val_bce, marker="s", label="ConvHQVAE BCE")
axes[1].set_title("BCE de validacion")
axes[1].set_xlabel("Epoca"); axes[1].set_ylabel("BCE")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2. Comparacion visual de reconstrucciones

In [ ]:
def show_reconstructions(model_cvae, model_hqvae, dataset, device, n_images=8):
    model_cvae.eval()
    model_hqvae.eval()

    noisy = torch.stack([dataset[i][0] for i in range(n_images)]).to(device)
    clean = torch.stack([dataset[i][1] for i in range(n_images)])

    with torch.no_grad():
        recon_cvae = model_cvae.reconstruct(noisy).cpu()
        recon_hqvae = model_hqvae.reconstruct(noisy).cpu()

    noisy = noisy.cpu()

    rows = ["Noisy input", "Clean", "ConvVAE", "ConvHQVAE"]
    data = [noisy, clean, recon_cvae, recon_hqvae]

    fig, axes = plt.subplots(4, n_images, figsize=(2 * n_images, 8.5))

    for r in range(4):
        for i in range(n_images):
            axes[r, i].imshow(to_disp(data[r][i]), cmap=CMAP)
            axes[r, i].axis("off")
        axes[r, 0].set_title(rows[r], loc="left", fontsize=10)

    plt.suptitle("ConvVAE vs ConvHQVAE sobre CIFAR-100\n(Ruido: Gaussian + Salt-Pepper)", fontsize=14)
    plt.tight_layout()
    plt.show()


show_reconstructions(model_cvae, model_hqvae, test_dataset_dn, device, n_images=8)

### 3.3. Metricas cuantitativas

In [ ]:
from sklearn.metrics import mean_squared_error


def evaluate_vae(model, dataset, device):
    model.eval()
    mse_list, psnr_list = [], []

    with torch.no_grad():
        for noisy, clean in DataLoader(dataset, batch_size=32, shuffle=False):
            noisy, clean = noisy.to(device), clean.to(device)
            recon = model.reconstruct(noisy)

            mse = F.mse_loss(recon, clean, reduction='none')
            mse = mse.view(mse.size(0), -1).mean(dim=1)
            mse_list.extend(mse.cpu().numpy())

            psnr = 10 * np.log10(1.0 / (mse.cpu().numpy() + 1e-10))
            psnr_list.extend(psnr)

    return {"MSE": np.mean(mse_list), "PSNR": np.mean(psnr_list)}

In [ ]:
from skimage.metrics import structural_similarity as ssim


def ssim_dataset(model, loader, device):
    """SSIM medio sobre el loader. `channel_axis` se activa solo en RGB."""
    model.eval()
    vals = []

    with torch.no_grad():
        for noisy, clean in loader:
            noisy, clean = noisy.to(device), clean.to(device)
            recon = model.reconstruct(noisy)

            recon_np = recon.cpu().numpy()
            clean_np = clean.cpu().numpy()

            for i in range(recon_np.shape[0]):
                r, c = recon_np[i], clean_np[i]
                if IN_CH == 1:
                    vals.append(ssim(c[0], r[0], data_range=1.0))
                else:
                    vals.append(
                        ssim(
                            np.transpose(c, (1, 2, 0)),
                            np.transpose(r, (1, 2, 0)),
                            data_range=1.0,
                            channel_axis=-1,
                        )
                    )

    return float(np.mean(vals))

In [ ]:
cvae_results = evaluate_vae(model_cvae, test_dataset_dn, device)
hqvae_results = evaluate_vae(model_hqvae, test_dataset_dn, device)

cvae_ssim = ssim_dataset(model_cvae, test_loader_dn, device)
hqvae_ssim = ssim_dataset(model_hqvae, test_loader_dn, device)

print("Classical VAE:", cvae_results, "SSIM:", round(cvae_ssim, 4))
print("Hybrid HQVAE :", hqvae_results, "SSIM:", round(hqvae_ssim, 4))

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model":   ["Classical ConvVAE", "Hybrid ConvHQVAE"],
    "Dataset": ["CIFAR-100", "CIFAR-100"],
    "Noise":   ["Gaussian + Salt-Pepper", "Gaussian + Salt-Pepper"],
    "MSE":     [cvae_results["MSE"], hqvae_results["MSE"]],
    "PSNR":    [cvae_results["PSNR"], hqvae_results["PSNR"]],
    "SSIM":    [cvae_ssim, hqvae_ssim],
})

comparison

### 3.4. Notas para la memoria

A diferencia de los notebooks de MNIST, aqui **si** se calcula SSIM ademas de
MSE y PSNR, de modo que las tres metricas son directamente comparables con las
tablas del capitulo de resultados.

Al interpretar estos numeros conviene recordar dos cosas:

1. El ruido es **fijo** (Gaussian + Salt-Pepper), no sorteado por imagen. Los valores no son
   por tanto comparables con los del barrido de ruido mixto aleatorio.
2. CIFAR-100 tiene mucha mas variabilidad de contenido que MNIST con el mismo
   presupuesto de 300 imagenes de entrenamiento, asi que es esperable un MSE
   mayor y un PSNR menor en ambos modelos. Lo relevante no es el valor
   absoluto sino **si la diferencia entre el modelo clasico y el hibrido se
   mantiene, se amplia o cambia de signo** respecto a MNIST.

### 3.5. Exportacion a HTML

In [ ]:
import sys
import subprocess
from pathlib import Path

notebook = Path("vcae_VS_hqvae_gaussian_saltpepper.ipynb").resolve()
output_dir = notebook.parent

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "nbconvert",
        "--to", "html",
        str(notebook),
        "--output-dir", str(output_dir),
    ],
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---")
print(result.stdout)
print("\n--- STDERR ---")
print(result.stderr)

if result.returncode == 0:
    print("\nHTML generado correctamente:")
    print(output_dir / "vcae_VS_hqvae_gaussian_saltpepper.html")